In [4]:
import nflreadpy as nfl
import polars as pl
import numpy as np
import torch
import data_cleaning
import schedule_formatting
import sklearn as skl


TRAINING_SEASONS = [x for x in range(2020, 2026)]
TRAINING_FEATURES = ['week', 'location', 'away_rest', 'home_rest',
                     'home_WR_0', 'home_WR_1', 'home_DB_0', 'home_DB_1',
                     'home_DB_2', 'home_DB_3', 'home_DL_0', 'home_DL_1',
                     'home_DL_2', 'home_DL_3', 'home_TE_0', 'home_K_0',
                     'home_LB_0', 'home_LB_1', 'home_LB_2', 'home_RB_0',
                     'home_RB_1', 'home_QB_0', 'away_WR_0', 'away_WR_1',
                     'away_DB_0', 'away_DB_1', 'away_DB_2', 'away_DB_3',
                     'away_DL_0', 'away_DL_1', 'away_DL_2', 'away_DL_3',
                     'away_TE_0', 'away_K_0', 'away_LB_0', 'away_LB_1',
                     'away_LB_2', 'away_RB_0', 'away_RB_1', 'away_QB_0']
TRAINING_LABELS = ['away_score', 'home_score', 'result', 'total']


# A list of abbreviations for positions on a football team.
POSITIONS: list[str] = ["WR", "DB", "DL", "TE", "K", "LB", "RB", "QB"]


# A dictionary consisting of abbreviations for positions, as 
# well as the number of players on the field that have that
# position at a given time.
POSITION_AMOUNTS: dict[str, int] = {"WR": 2, "OL": 5, "DB": 4, "DL": 4,
                                    "TE": 1, "K": 1, "LB": 3, "LS": 1,
                                    "RB": 2, "P": 1, "QB": 1}


# Schemas for the columns that will be added to the DataFrame of
# a given NFL match.
HOME_R_SCHEMA = pl.Struct({
    f"home_{position}_{i}": pl.Float64
    for position in POSITIONS
    for i in range(POSITION_AMOUNTS[position])
})
AWAY_R_SCHEMA = pl.Struct({
    f"away_{position}_{i}": pl.Float64
    for position in POSITIONS
    for i in range(POSITION_AMOUNTS[position])
})


# A list of the columns that are relevant for the add_roster
# function
R_COLS = ["home_team", "away_team",
                   "year", "month", "day", "week", "season"]


# A DataFrame of match schedules for the seasons of interest
SCHEDULES = data_cleaning.make_schedules(TRAINING_SEASONS)

In [5]:
rosters = schedule_formatting.add_home_and_away(SCHEDULES.slice(0, 200))
rosters.head()

game_id,season,game_type,week,gametime,away_team,away_score,home_team,home_score,location,result,total,overtime,away_rest,home_rest,away_moneyline,home_moneyline,spread_line,away_spread_odds,home_spread_odds,total_line,under_odds,over_odds,year,month,day,home_WR_0,home_WR_1,home_DB_0,home_DB_1,home_DB_2,home_DB_3,home_DL_0,home_DL_1,home_DL_2,home_DL_3,home_TE_0,home_K_0,home_LB_0,home_LB_1,home_LB_2,home_RB_0,home_RB_1,home_QB_0,away_WR_0,away_WR_1,away_DB_0,away_DB_1,away_DB_2,away_DB_3,away_DL_0,away_DL_1,away_DL_2,away_DL_3,away_TE_0,away_K_0,away_LB_0,away_LB_1,away_LB_2,away_RB_0,away_RB_1,away_QB_0
str,i32,str,i32,str,str,i32,str,i32,str,i32,i32,i32,i32,i32,i32,i32,f64,i32,i32,f64,i32,i32,i64,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""2020_01_HOU_KC""",2020,"""REG""",1,"""20:20""","""HOU""",20,"""KC""",34,"""Home""",14,54,0,7,7,349,-423,9.5,-105,-105,53.5,-109,-102,2020,9,10,2.9,3.6,13.1,24.7,34.0,42.8,13.9,16.8,19.7,30.0,1.0,1.3,50.1,55.2,68.3,1.0,6.3,1.1,25.1,38.8,1.0,1.0,1.1,2.5,8.7,39.4,342.0,455.0,24.5,7.8,14.7,15.8,18.7,17.7,29.4,3.0
"""2020_01_SEA_ATL""",2020,"""REG""",1,"""13:00""","""SEA""",38,"""ATL""",25,"""Home""",-13,63,0,7,7,102,-112,1.0,-105,-105,49.5,-108,-103,2020,9,13,3.2,5.3,9.5,39.2,51.5,55.7,19.7,22.7,23.3,30.0,8.6,12.5,10.6,16.0,21.7,13.5,15.9,7.9,16.8,18.3,1.7,2.0,15.6,34.3,40.5,48.4,52.0,73.0,13.5,8.4,1.8,1.9,7.3,15.1,23.7,3.3
"""2020_01_CLE_BAL""",2020,"""REG""",1,"""13:00""","""CLE""",6,"""BAL""",38,"""Home""",32,44,0,7,7,260,-302,7.0,-107,-103,47.0,100,-112,2020,9,13,28.8,30.7,29.4,34.6,40.0,47.9,11.3,18.5,51.7,76.6,2.9,1.5,23.3,44.4,53.2,4.1,13.2,1.0,13.2,16.5,29.0,30.3,31.2,41.7,5.0,19.4,34.0,35.0,5.3,22.3,45.8,90.0,554.0,9.0,10.0,11.1
"""2020_01_NYJ_BUF""",2020,"""REG""",1,"""13:00""","""NYJ""",17,"""BUF""",27,"""Home""",10,44,0,7,7,242,-279,6.5,-110,100,39.5,-114,102,2020,9,13,11.0,22.0,6.7,16.0,25.5,28.5,26.5,33.5,43.8,53.0,24.5,18.6,17.7,21.3,21.3,18.4,20.8,7.6,36.3,37.9,24.6,29.5,32.2,36.5,64.5,71.0,338.0,341.0,46.3,27.9,38.7,45.6,51.3,12.0,21.3,17.5
"""2020_01_LV_CAR""",2020,"""REG""",1,"""13:00""","""LV""",34,"""CAR""",30,"""Home""",-4,64,0,7,7,-124,134,-3.0,-112,-113,48.0,-101,-110,2020,9,13,7.6,9.0,31.5,36.0,45.6,49.0,35.5,36.9,47.0,51.3,19.6,14.9,16.7,17.5,27.3,1.0,1.0,21.9,10.4,18.1,17.1,20.0,51.0,58.6,20.0,20.3,40.0,45.3,5.2,20.4,4.3,5.5,7.3,6.1,8.3,20.6


In [3]:
print(rosters.columns)

['game_id', 'season', 'game_type', 'week', 'gametime', 'away_team', 'away_score', 'home_team', 'home_score', 'location', 'result', 'total', 'overtime', 'away_rest', 'home_rest', 'away_moneyline', 'home_moneyline', 'spread_line', 'away_spread_odds', 'home_spread_odds', 'total_line', 'under_odds', 'over_odds', 'year', 'month', 'day', 'home_WR_0', 'home_WR_1', 'home_DB_0', 'home_DB_1', 'home_DB_2', 'home_DB_3', 'home_DL_0', 'home_DL_1', 'home_DL_2', 'home_DL_3', 'home_TE_0', 'home_K_0', 'home_LB_0', 'home_LB_1', 'home_LB_2', 'home_RB_0', 'home_RB_1', 'home_QB_0', 'away_WR_0', 'away_WR_1', 'away_DB_0', 'away_DB_1', 'away_DB_2', 'away_DB_3', 'away_DL_0', 'away_DL_1', 'away_DL_2', 'away_DL_3', 'away_TE_0', 'away_K_0', 'away_LB_0', 'away_LB_1', 'away_LB_2', 'away_RB_0', 'away_RB_1', 'away_QB_0']


In [9]:
def clean_for_training(data: pl.DataFrame) -> pl.DataFrame:
    """
    Modifies the columns of a DataFrame data containing
    match schedules to make the columns usable for training
    a machine learning model
    """
    df = data.drop_nulls().with_columns(pl.col("location").replace({"Home": 1, "Away": 0}).cast(int))

    return df

In [14]:
cleaned_rosters = clean_for_training(rosters)
cleaned_rosters.shape

(127, 62)

In [16]:
features = cleaned_rosters[TRAINING_FEATURES].slice(0, 100)
labels = cleaned_rosters[TRAINING_LABELS].slice(0, 100)
model = skl.tree.DecisionTreeRegressor()
model.fit(features, labels)

,"criterion criterion: {""squared_error"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""absolute_error"" for the meanabsolute error, which minimizes the L1 loss using the median of each terminalnode, and ""poisson"" which uses reduction in Poisson deviance to find splits,also using the mean of each terminal node... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 0.24 Poisson deviance criterion... versionchanged:: 1.9 Criterion `""friedman_mse""` was deprecated.",'squared_error'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split among considered features for this split.",'best'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.For an example of how ``max_depth`` influences the model, see:ref:`sphx_glr_auto_examples_tree_plot_tree_regression.py`.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`.Note: splitting may inspect more than ``max_features`` features ifneeded to find a valid split.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary <random_state>` for details.",None
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow a tree wi

In [17]:
predictions = pl.DataFrame(model.predict(cleaned_rosters[TRAINING_FEATURES].slice(100, 27)), schema=TRAINING_LABELS)
actuals = cleaned_rosters[TRAINING_LABELS].slice(100, 27)

skl.metrics.mean_squared_error(predictions["home_score"], actuals["home_score"])

130.66666666666666

In [18]:
print(predictions)
print(actuals)

shape: (27, 4)
┌────────────┬────────────┬────────┬───────┐
│ away_score ┆ home_score ┆ result ┆ total │
│ ---        ┆ ---        ┆ ---    ┆ ---   │
│ f64        ┆ f64        ┆ f64    ┆ f64   │
╞════════════╪════════════╪════════╪═══════╡
│ 24.0       ┆ 20.0       ┆ -4.0   ┆ 44.0  │
│ 34.0       ┆ 17.0       ┆ -17.0  ┆ 51.0  │
│ 46.0       ┆ 23.0       ┆ -23.0  ┆ 69.0  │
│ 27.0       ┆ 25.0       ┆ -2.0   ┆ 52.0  │
│ 17.0       ┆ 28.0       ┆ 11.0   ┆ 45.0  │
│ …          ┆ …          ┆ …      ┆ …     │
│ 20.0       ┆ 24.0       ┆ 4.0    ┆ 44.0  │
│ 30.0       ┆ 35.0       ┆ 5.0    ┆ 65.0  │
│ 17.0       ┆ 27.0       ┆ 10.0   ┆ 44.0  │
│ 31.0       ┆ 13.0       ┆ -18.0  ┆ 44.0  │
│ 13.0       ┆ 27.0       ┆ 14.0   ┆ 40.0  │
└────────────┴────────────┴────────┴───────┘
shape: (27, 4)
┌────────────┬────────────┬────────┬───────┐
│ away_score ┆ home_score ┆ result ┆ total │
│ ---        ┆ ---        ┆ ---    ┆ ---   │
│ i32        ┆ i32        ┆ i32    ┆ i32   │
╞════════════╪═══════════